# Chapter 3: Implicit Differentiation Through a Voigt Fit

## What We Are Doing

We want to optimize a parameter `mu` that generates data. The data goes through a Voigt fit which produces `gamma_fit`. Loss = (gamma_fit - target)^2.

**The problem:** The Voigt fit is an iterative process (L-BFGS). We cannot backpropagate *through* it.

**The solution:** Implicit differentiation. Instead of tracing the optimizer path, we differentiate *around* it using the optimality condition.

In this notebook we build this up step by step with a simple example.

---
## The Pseudo-Voigt Function

A Voigt profile is the convolution of a Gaussian and a Lorentzian. The **pseudo-Voigt** approximates this as a weighted sum:

$V(x) = \eta \cdot G(x) + (1-\eta) \cdot L(x)$

This is fully differentiable in PyTorch.

In [ ]:
import torch
import matplotlib.pyplot as plt

torch.manual_seed(42)

def pseudo_voigt(x, gamma, sigma_g, center, amplitude, eta):
    """Pseudo-Voigt: weighted sum of Gaussian and Lorentzian."""
    gamma = torch.abs(gamma)  # keep gamma positive
    gauss = amplitude * torch.exp(-0.5 * ((x - center) / sigma_g) ** 2) / \
            (sigma_g * torch.sqrt(torch.tensor(2 * torch.pi)))
    lorentz = amplitude * (gamma / torch.pi) / ((x - center) ** 2 + gamma ** 2)
    return eta * gauss + (1 - eta) * lorentz

# Visualize
x = torch.linspace(-10, 10, 200)
y = pseudo_voigt(x, torch.tensor([2.0]), torch.tensor([1.0]), 
                 torch.tensor([0.0]), torch.tensor([50.0]), torch.tensor([0.3]))

plt.plot(x.numpy(), y.numpy())
plt.xlabel('x'); plt.ylabel('Amplitude')
plt.title('Pseudo-Voigt (gamma=2, sigma=1, eta=0.3)')
plt.grid(alpha=0.3)
plt.show()
print('Ready')

---
## Step 1: Generate Data and Fit a Voigt

We create noisy Voigt data with `gamma_true=3`. Then fit a Voigt to recover it.

In [ ]:
x = torch.linspace(-10, 10, 100)

gamma_true = torch.tensor([3.0])
y_clean = pseudo_voigt(x, gamma_true, torch.tensor([1.0]), torch.tensor([0.0]),
                       torch.tensor([50.0]), torch.tensor([0.3]))
y_noisy = y_clean + torch.randn(100) * 0.3

# Fit
g = torch.tensor([5.0], requires_grad=True)
c = torch.tensor([0.5], requires_grad=True)
a = torch.tensor([40.0], requires_grad=True)

opt = torch.optim.LBFGS([g, c, a], max_iter=200, lr=0.5)

def closure():
    opt.zero_grad()
    y_pred = pseudo_voigt(x.detach(), g, torch.tensor([1.0]), c, a, torch.tensor([0.3]))
    loss = torch.mean((y_pred - y_noisy.detach()) ** 2)
    loss.backward()
    return loss

opt.step(closure)
print(f'True gamma:  3.0')
print(f'Fitted gamma: {g.item():.4f}')

# Visualize
y_fit = pseudo_voigt(x, g.detach(), torch.tensor([1.0]), 
                     c.detach(), a.detach(), torch.tensor([0.3]))
plt.plot(x.numpy(), y_noisy.numpy(), '.', alpha=0.5, label='Data')
plt.plot(x.numpy(), y_fit.detach().numpy(), '-', label='Fit')
plt.legend(); plt.grid(alpha=0.3)
plt.title(f'Fit: gamma = {g.item():.3f}')
plt.show()

---
## Step 2: Implicit Differentiation

### The Key Idea

At the optimal fit parameters $\theta^* = (\gamma^*, c^*, a^*)$, the gradient of the inner loss is zero:

$\frac{\partial L_{\text{inner}}}{\partial \theta} \big|_{\theta^*} = 0$

If we change $\gamma_{\text{true}}$ (which changes our data), $\theta^*$ shifts. Differentiating the zero-gradient condition:

$\frac{d}{d\gamma_{\text{true}}} \left( \frac{\partial L_{\text{inner}}}{\partial \theta} \right) = 0$

$\frac{\partial^2 L_{\text{inner}}}{\partial \theta^2} \cdot \frac{d\theta^*}{d\gamma_{\text{true}}} + \frac{\partial^2 L_{\text{inner}}}{\partial \theta \partial \gamma_{\text{true}}} = 0$

$\boxed{\frac{d\theta^*}{d\gamma_{\text{true}}} = -\mathbf{H}^{-1} \cdot \frac{\partial^2 L_{\text{inner}}}{\partial \theta \partial \gamma_{\text{true}}}}$

where $\mathbf{H} = \partial^2 L/\partial \theta^2$ is the Hessian.

**We never need to backprop through the L-BFGS iterations.** We only need the Hessian and the mixed derivative at the final optimum.

In [ ]:
# ---- Step 2a: Generate data WITH gradient tracking on gamma_true ----
gamma_true = torch.tensor([3.0], requires_grad=True)
y_data = pseudo_voigt(x, gamma_true, torch.tensor([1.0]), torch.tensor([0.0]),
                      torch.tensor([50.0]), torch.tensor([0.3]))
y_data = y_data + torch.randn(100) * 0.3  # noise (no grad needed for this line)

# ---- Step 2b: Fit (no gradient tracking here) ----
g = torch.tensor([5.0], requires_grad=True)
c = torch.tensor([0.5], requires_grad=True)
a = torch.tensor([40.0], requires_grad=True)
opt = torch.optim.LBFGS([g, c, a], max_iter=200, lr=0.5)

def closure():
    opt.zero_grad()
    y_pred = pseudo_voigt(x.detach(), g, torch.tensor([1.0]), c, a, torch.tensor([0.3]))
    loss = torch.mean((y_pred - y_data.detach()) ** 2)
    loss.backward()
    return loss

opt.step(closure)
print(f'After fit: gamma = {g.item():.4f}')

# ---- Step 2c: Implicit differentiation ----
# Create a fresh tensor at the optimum (with grad tracking)
g_opt = g.detach().clone().requires_grad_(True)

# Compute the inner loss again, now connecting:
# - g_opt (requires_grad) -> for the Hessian
# - y_data (depends on gamma_true through the data gen) -> for the mixed derivative
loss_inner = torch.mean(
    (pseudo_voigt(x.detach(), g_opt, torch.tensor([1.0]), torch.tensor([0.0]),
                  torch.tensor([50.0]), torch.tensor([0.3])) - y_data) ** 2
)

# First derivative: dL/dg
grad_g = torch.autograd.grad(loss_inner, g_opt, create_graph=True)[0]
print(f'Gradient at optimum (should be ~0): {grad_g.item():.6f}')

# Hessian: d2L/dg2 (second derivative)
H = torch.autograd.grad(grad_g, g_opt, retain_graph=True)[0]

# Mixed derivative: d2L/(dg * dgamma_true)
# grad_g depends on gamma_true THROUGH y_data
mixed = torch.autograd.grad(grad_g, gamma_true, retain_graph=True)[0]

# Final result: dgamma_fit / dgamma_true = -H^(-1) * mixed
dgamma_dgamma_true = -mixed / H

print(f'\nHessian H = {H.item():.4f}')
print(f'Mixed derivative = {mixed.item():.4f}')
print(f'dgamma_fit / dgamma_true = {dgamma_dgamma_true.item():.4f}')
print()
print('Interpretation: if gamma_true increases by 1, gamma_fit increases by ~')
print(f'{dgamma_dgamma_true.item():.2f}. This is the gradient through the fit.')

---
## What Just Happened

We computed $d\gamma_{\text{fit}} / d\gamma_{\text{true}}$ **without backpropagating through the L-BFGS optimizer**.

The key numbers:
- **Hessian H** = $\partial^2 L / \partial \gamma^2$ at the optimum (curvature of the loss landscape)
- **Mixed derivative** = $\partial^2 L / (\partial \gamma \partial \gamma_{\text{true}})$ (how the gradient at optimum changes when data changes)
- The ratio gives us the sensitivity of the fit result to the input

### Why This Matters for Our Project

In the full pipeline, data comes from sampling `N(mu, 1)` and squaring. The chain rule becomes:

$\frac{dL}{d\mu} = \frac{dL}{d\gamma_{\text{fit}}} \cdot \frac{d\gamma_{\text{fit}}}{d\gamma_{\text{true}}} \cdot \frac{d\gamma_{\text{true}}}{d\mu}$

Implicit differentiation gives us the middle term. The rest is standard autograd.